# 10 — Retraining Strategies

## What
Retraining strategies determine *when* and *how* to retrain a production model. Options range from full periodic retraining to incremental online learning. The right strategy depends on how fast the data distribution changes, how expensive retraining is, and how much label latency exists.

## Why
Retraining too rarely lets models degrade silently. Retraining too often wastes compute and risks instability from noisy label signals. A triggered pipeline — retrain when PSI exceeds a threshold or performance drops below a floor — balances responsiveness with cost.

## Community context
Four main strategies: (1) *Full periodic retraining* — simple, safe baseline; (2) *Triggered retraining* — drift or performance threshold triggers a pipeline run; (3) *Online learning* — model updates on each batch without full retraining (SGD-style, effective for structured/tabular data); (4) *Continual learning* — avoids catastrophic forgetting via EWC, replay buffers, or progressive networks (dominant in deep learning research).

In [ ]:
# Retraining strategy comparison framework
import numpy as np
from collections import deque

class RetrainingSimulator:
    """
    Simulates model performance under different retraining strategies
    as the data distribution gradually shifts.
    """
    def __init__(self, n_periods=24):
        self.n_periods = n_periods
        # True underlying AUC if model was always up to date
        self.oracle_auc = 0.88
        # Degradation per period without retraining
        self.decay_per_period = 0.008
        # Recovery after retraining (instantaneous)
        self.recovery = 0.90

    def simulate_no_retraining(self):
        auc = self.oracle_auc
        history = []
        for t in range(self.n_periods):
            noise = np.random.normal(0, 0.005)
            auc = max(0.5, auc - self.decay_per_period + noise)
            history.append(auc)
        return history, 0

    def simulate_periodic(self, period=6):
        auc = self.oracle_auc
        history = []
        retrains = 0
        for t in range(self.n_periods):
            if t > 0 and t % period == 0:
                auc = self.recovery
                retrains += 1
            noise = np.random.normal(0, 0.005)
            auc = max(0.5, auc - self.decay_per_period + noise)
            history.append(auc)
        return history, retrains

    def simulate_triggered(self, floor=0.82):
        auc = self.oracle_auc
        history = []
        retrains = 0
        for t in range(self.n_periods):
            noise = np.random.normal(0, 0.005)
            auc = max(0.5, auc - self.decay_per_period + noise)
            history.append(auc)
            if auc < floor:
                auc = self.recovery
                retrains += 1
        return history, retrains

    def simulate_online_learning(self, update_gain=0.004):
        """Online update partially compensates for drift each period"""
        auc = self.oracle_auc
        history = []
        for t in range(self.n_periods):
            noise = np.random.normal(0, 0.003)
            # Online update recovers some of the decay
            net_change = -self.decay_per_period + update_gain + noise
            auc = np.clip(auc + net_change, 0.5, 0.92)
            history.append(auc)
        return history, self.n_periods  # updates every period

np.random.seed(42)
sim = RetrainingSimulator(n_periods=24)

strategies = [
    ('No retraining',       *sim.simulate_no_retraining()),
    ('Periodic (6-month)',   *sim.simulate_periodic(6)),
    ('Triggered (floor=0.82)',*sim.simulate_triggered(0.82)),
    ('Online learning',      *sim.simulate_online_learning()),
]

print(f'{"Strategy":<28} {"Avg AUC":>8} {"Min AUC":>8} {"Retrains":>10}')
print('-' * 58)
for name, history, retrains in strategies:
    print(f'{name:<28} {np.mean(history):>8.4f} {np.min(history):>8.4f} {retrains:>10}')

## Catastrophic Forgetting in Continual Learning

When neural networks are fine-tuned on new data, they forget old knowledge — *catastrophic forgetting*. Elastic Weight Consolidation (EWC) adds a regularisation term to protect important weights. This is the foundation of continual learning.

In [ ]:
# EWC regularisation concept demo (tabular proxy)
def ewc_loss(new_loss, old_params, new_params, fisher_diagonal, lambda_ewc=100):
    """
    EWC total loss = task loss + lambda * sum_i F_i * (theta_i - theta_star_i)^2
    Fisher diagonal approximates parameter importance from old task.
    """
    penalty = sum(
        fisher_diagonal[i] * (new_params[i] - old_params[i])**2
        for i in range(len(old_params))
    )
    return new_loss + lambda_ewc * penalty

# Simulate: 5 parameters, old task learned important weights
np.random.seed(1)
old_params = np.array([0.8, -0.3, 1.2, 0.1, -0.9])
fisher_diag = np.array([10., 0.1, 5., 0.5, 8.])  # high = important

# Naive update: moves all params toward new task optimum
naive_new  = old_params + np.random.randn(5) * 0.5
# EWC update: stays close to old params on high-Fisher dimensions
ewc_new    = old_params.copy()
for i in range(5):
    # EWC gradient: gradient of penalty pulls back toward old_params
    gradient_penalty = 2 * fisher_diag[i] * (ewc_new[i] - old_params[i])
    ewc_new[i] -= 0.05 * (np.random.randn() + 0.1 * gradient_penalty)

print('Parameter changes from old task optimal:')
print(f'{"Param":<8} {"Fisher":>8} {"Naive change":>14} {"EWC change":>12}')
for i in range(5):
    print(f'  w{i}    {fisher_diag[i]:>8.1f} {naive_new[i]-old_params[i]:>14.4f} {ewc_new[i]-old_params[i]:>12.4f}')
print('\nHigh-Fisher parameters are better protected by EWC.')